In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
nasal = pd.read_csv("Flow - 30-05-2024.txt", delimiter='\t')
nasal = nasal.iloc[5: , :]

nasal['Time'] = nasal['Signal Type: Flow_TH_Type'].str.split(';').str[0]
nasal['Value'] = nasal['Signal Type: Flow_TH_Type'].str.split(';').str[1]

nasal_origin = nasal.copy()

nasal.drop(['Signal Type: Flow_TH_Type'], axis=1, inplace=True)
nasal = nasal.set_index('Time')

nasal.index = pd.to_datetime(nasal.index, format="%d.%m.%Y %H:%M:%S,%f")
nasal['Value'] = pd.to_numeric(nasal['Value'].astype(str).str.strip(), errors='coerce')

In [52]:
events = pd.read_csv("Flow Events - 30-05-2024.txt", delimiter='\t')
events = events.iloc[3: , :]
events['Time'] = events[r'Signal ID: FlowD\flow'].str.split(';').str[0]
events['Value'] = events[r'Signal ID: FlowD\flow'].str.split(';').str[1]
events['Disorder'] = events[r'Signal ID: FlowD\flow'].str.split(';').str[2]
events['Stage'] = events[r'Signal ID: FlowD\flow'].str.split(';').str[3]
events_origin = events


events[['start_str', 'end_str']] = events['Time'].str.split('-', expand=True)
events['date_part'] = events['start_str'].str.split(' ').str[0]
events['end_str'] = events['date_part'] + ' ' + events['end_str']
events['start_time'] = pd.to_datetime(events['start_str'], format="%d.%m.%Y %H:%M:%S,%f")
events['end_time'] = pd.to_datetime(events['end_str'], format="%d.%m.%Y %H:%M:%S,%f")
events.drop([r'Signal ID: FlowD\flow', 'Time', 'start_str', 'end_str', 'date_part'], axis=1, inplace=True)

## creating window

In [53]:
signal = nasal['Value'].values
time_index = nasal.index

In [54]:
sampling_interval = (time_index[1] - time_index[0]).total_seconds()
samples_per_30sec = int(30 / sampling_interval)
samples_per_15sec = int(15 / sampling_interval)

print("Samples per 30 sec:", samples_per_30sec)
print("Samples per 15 sec:", samples_per_15sec)

Samples per 30 sec: 967
Samples per 15 sec: 483


In [55]:
windows = []
window_times = []

for start in range(0, len(signal) - samples_per_30sec + 1, samples_per_15sec):

    end = start + samples_per_30sec

    window_signal = signal[start:end]

    window_start_time = time_index[start]
    window_end_time = time_index[end - 1]

    windows.append(window_signal)
    window_times.append((window_start_time, window_end_time))

X = np.array(windows)

In [56]:
print("Shape:", X.shape)

Shape: (1810, 967)


## Labelling windows

In [ ]:
labels = []

In [58]:
for (window_start, window_end) in window_times:

    overlapping = events[
        (events['start_time'] <= window_end) &
        (events['end_time'] >= window_start)
    ]

    max_overlap = 0
    assigned_label = "Normal"

    for _, row in overlapping.iterrows():

        overlap_start = max(window_start, row['start_time'])
        overlap_end = min(window_end, row['end_time'])

        overlap_duration = (overlap_end - overlap_start).total_seconds()

        if overlap_duration > max_overlap:
            max_overlap = overlap_duration
            assigned_label = row['Disorder']

    if max_overlap < 15:
        assigned_label = "Normal"

    labels.append(assigned_label)

In [59]:
label_map = {
    "Normal": 0,
    "Hypopnea": 1,
    "Obstructive Apnea": 2
}

y = np.array([label_map[l] for l in labels])

In [60]:
print(len(y), X.shape[0])

1810 1810


In [61]:
from collections import Counter
print(Counter(y))

Counter({np.int64(0): 1716, np.int64(1): 79, np.int64(2): 15})
